In [44]:
import torch
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import re
from datasets import load_dataset
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
#from cleantext import clean
#import pytorch_lightning as pl
from datasets import Dataset as dts
#from sentence_transformers import SentenceTransformer
from torch.nn.utils.rnn import pack_padded_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn.utils.rnn import pad_sequence
from torchvision.ops import focal_loss
from torch.utils.data import WeightedRandomSampler

In [45]:
!uv pip install optuna

Using Python 3.12.12 environment at: /home/ydlawq/torch_rocm_v/.venv
Audited 1 package in 39ms


In [46]:
import logging
from pathlib import Path


In [47]:
!pip install mawo-razdel emoji

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: /usr/bin/python3 -m pip install --upgrade pip


In [48]:
import numpy as np
import emoji

In [49]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [50]:
dataset = load_dataset('astromis/presuicidal_signals')


In [51]:
ds_train = dataset['train'].to_pandas()
ds_test = dataset['test'].to_pandas()

In [52]:
import re

In [53]:
emoji_pattern = re.compile(r"<emoji>(.*?)</emoji>")

ds_train["emoji_count"] = ds_train["text"].str.count(r"<emoji>.*?</emoji>")
ds_train["has_emoji"] = ds_train["emoji_count"] > 0
emoji_ds_train = ds_train[ds_train["has_emoji"]]
total_texts = len(ds_train)
texts_with_emoji = len(emoji_ds_train)
positive_texts = (emoji_ds_train["label"] == 1).sum()
positive_share = (positive_texts / texts_with_emoji if texts_with_emoji > 0 else 0)


print(f"Всего текстов: {total_texts}")
print(f"Текстов с эмодзи: {texts_with_emoji}")
print(f"Доля текстов с эмодзи: {texts_with_emoji / total_texts:.2%}")
print(f"Положительных текстов с эмодзи: {positive_texts}")
print(f"Доля положительного класса среди текстов с эмодзи: {positive_share:.2%}")





Всего текстов: 22787
Текстов с эмодзи: 2949
Доля текстов с эмодзи: 12.94%
Положительных текстов с эмодзи: 283
Доля положительного класса среди текстов с эмодзи: 9.60%


In [54]:
print(
    emoji_ds_train["label"]
    .value_counts()
    .rename("count")
)

label
0    2666
1     283
Name: count, dtype: int64


In [55]:
def emoji_to_clean_russian(text):
    clean_tags = re.sub(r'<emoji>\s*(.*?)\s*</emoji>', lambda m: f":{m.group(1).strip().replace(' ', '_').lower()}:", text)
    emojified = emoji.emojize(clean_tags, language='en')
    

    ru_demojized = emoji.demojize(emojified, language='ru')
    

    final_text = ru_demojized.replace('::', ' ').replace(':', ' ').replace('_', ' ')
    final_text = re.sub(r'\s+', ' ', final_text).strip()
    final_text = re.sub(r'[,;:«»""()\[\]{}]', ' ', final_text)
    return final_text

In [56]:
ds_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 22787 entries, 0 to 22786
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   text         22787 non-null  str  
 1   label        22787 non-null  int64
 2   emoji_count  22787 non-null  int64
 3   has_emoji    22787 non-null  bool 
dtypes: bool(1), int64(2), str(1)
memory usage: 4.1 MB


In [57]:
print(f'MEAN LABEL ON TRAIN: {ds_train['label'].mean()}')
print(f'MEAN LABEL ON TEST: {ds_test['label'].mean()}')

MEAN LABEL ON TRAIN: 0.15017334445078334
MEAN LABEL ON TEST: 0.15542131667861164


In [58]:
print(ds_train.isna().sum())
print(ds_test.isna().sum())

text           0
label          0
emoji_count    0
has_emoji      0
dtype: int64
text     0
label    0
dtype: int64


In [59]:
def cleaning_text(text):
    text = re.sub(r"</?emoji>", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [60]:
ds_test['text'] = ds_test['text'].apply(emoji_to_clean_russian)
ds_train['text'] = ds_train['text'].apply(emoji_to_clean_russian)

In [61]:
ds_train['text_len'] = ds_train['text'].str.len()
ds_test['text_len'] = ds_test['text'].str.len()

In [62]:
print(ds_train['text_len'].mean())
print(ds_train['text_len'].min())
print(ds_train['text_len'].max())

90.8253828937552
0
2582


In [63]:
ds_train['count_words'] = ds_train['text'].str.split().str.len()
ds_test['count_words'] = ds_test['text'].str.split().str.len()

In [64]:
print(ds_train['count_words'].mean())
print(ds_train['count_words'].min())
print(ds_train['count_words'].max())

15.44753587571861
0
421


In [65]:
print(len(ds_train[ds_train['count_words'] == 2]))
print(len(ds_test[ds_test['count_words'] == 0]))

923
9


In [66]:
ds_train[ds_train['count_words'] ==2]

,text,label,emoji_count,has_emoji,text_len,count_words
6,слава кпсс!!,0,0,False,12,2
12,Давайте поболтаем,0,0,False,17,2
18,Есть желающие?#закос,0,0,False,20,2
21,люблю герлловвввер,0,0,False,18,2
41,Я ЕБАОА,0,0,False,7,2
...,...,...,...,...,...,...
22592,изгибы тельцА,0,0,False,13,2
22596,паучье молоко,0,0,False,13,2
22672,я живая,0,0,False,7,2
22711,хочу каре...,0,0,False,12,2


In [67]:
ds_train['text'][333]

'бог подарил мне подарок в честь дня рождения стэй в лице человека который не смог пить пакетированный чай и сам взял заварил вкуснейший чай бля слезы рекой слезы рекой чел религия религия'

In [68]:
ds_train = ds_train[ds_train['count_words'] >=2]
len(ds_train)

22092

In [69]:
y_test = ds_test['label']
X_test = ds_test.drop(columns=['label'])

In [70]:
y = ds_train['label']
X = ds_train.drop(columns=['label'])


In [71]:
X_train, X_valid, y_train, y_valid = train_test_split(X,  y, test_size=0.05, stratify=y, random_state=42)

In [72]:
X_train.head()


,text,emoji_count,has_emoji,text_len,count_words
13049,я слова свои сдерживаю,0,False,22,4
11328,я больше никуда не пойду. это была 6. я не хоч...,0,False,247,42
19570,Так прикольно смотреть бсд тогда когда перечи...,0,False,170,24
20487,Блять. Голос моей мамы уже меня выводит меньше...,0,False,70,13
4174,Смотрите что нашла Кто помнит откуда опенинг?,0,False,46,7


In [73]:
from mawo_razdel import tokenize
from collections import Counter

In [74]:
from collections import Counter
import numpy as np


class RNNTokenizer:
    
    def __init__(self, tokenize_fn=None, min_freq=2, max_len=None, lowercase=True):
        self.tokenize_fn = tokenize_fn or (lambda s: s.split())
        self.min_freq = min_freq
        self.max_len = max_len
        self.lowercase = lowercase

        self.vocab = {'<PAD>': 0, '<UNK>': 1}
        self.pad_token_id = 0
        self.unk_token_id = 1

    def fit(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(self._tokenize(text))
        for token, freq in counter.most_common():
            if freq >= self.min_freq:
                self.vocab[token] = len(self.vocab)
        return self

    def fit_max_len(self, texts, pct=99):
        lengths = np.array([len(self._tokenize(t)) for t in texts])
        self.max_len = int(np.percentile(lengths, pct))
        return self.max_len, lengths.mean(), np.percentile(lengths, pct)

    def encode(self, text, **kwargs):
        output = {'input_ids': []}
        for t in text:
            tokens = self._tokenize(t)
            ids = [self.vocab.get(t, self.unk_token_id) for t in tokens]
            if self.max_len is not None:
                ids = ids[:self.max_len]
            output['input_ids'].append(ids)   
        return output

    def __call__(self, text):
        return self.encode(text)

    def _tokenize(self, text):
        tokens = self.tokenize_fn(text)
        return [t.text.lower() for t in tokens] if self.lowercase else list(tokens)

    def __len__(self):
        return len(self.vocab)

In [75]:
tokenizer_rnn = RNNTokenizer(tokenize_fn=tokenize)
tokenizer_rnn.fit_max_len(X_train['text'])
tokenizer_rnn.fit(X_train['text'])
vocab_size = len(tokenizer_rnn.vocab)

In [76]:
class MyDataset(Dataset):

    

    def __init__(self, input, tokenizer, labels, attention_mask=True):

        self.text = input.tolist()
        tokenized_text = tokenizer(self.text, padding=False, truncation=True, max_length=128, return_tensors=None)

        self.enc = tokenized_text
        

        self.labels = labels.tolist()
        
                

    def __len__(self):
        return len(self.labels)


    def __getitem__(self, index):
        item = {k: torch.tensor(v[index], dtype=torch.long) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item


In [78]:
def collate_fn(batch, pad_value):
    padding_value = pad_value

    sequences = [item["ids"] for item in batch]
    labels    = [item["labels"]    for item in batch]


    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=padding_value)
    labels = torch.stack(labels)
    return {
        'ids' : padded_sequences, 'labels': labels
    }
    


In [79]:
train_dataset = MyDataset(input=X_train['text'], labels=y_train, tokenizer=tokenizer_rnn.encode, attention_mask=False)
val_dataset = MyDataset(input=X_valid['text'], labels=y_valid, tokenizer=tokenizer_rnn.encode, attention_mask=False)
train_dataloader = DataLoader(train_dataset, collate_fn=collate_fn, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, collate_fn=collate_fn, batch_size=4, shuffle=True)

In [33]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_index=0):
        super().__init__()

        self.embeddings = nn.Embedding(
            num_embeddings=vocab_size,
            padding_idx=0,
            embedding_dim=embed_dim
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=dropout if num_layers >1 else 0.0
        )
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(hidden_dim* 2, 1)
        self.pad_index = pad_index

    def forward(self, input_ids):

        lengths = (input_ids != self.pad_index).sum(dim=-1)
        embeds = self.embeddings(input_ids) # -> [batch_size, seq_len, embed_dim]

        packed = pack_padded_sequence(embeds, lengths.cpu(), batch_first=True, enforce_sorted=False)


        _, (h_n, _) = self.lstm(packed)

        h = torch.cat([h_n[-2], h_n[-1]],dim=-1)

        logits = self.fc(self.dropout(h))

        return logits.squeeze(1-1)




In [34]:
from sklearn.metrics import recall_score, auc, f1_score, precision_recall_curve
from sklearn.metrics import roc_auc_score
import gc


gc.collect()

1037

In [35]:
class_counts = np.bincount(y_train)
class_weight = 1.0/ class_counts
sample_weights = class_weight[y_train]
sampler = WeightedRandomSampler(
weights=torch.from_numpy(sample_weights).double(), num_samples=len(sample_weights),
replacement=True
)
pos_weight = torch.tensor([((1 - y_train.mean()) / y_train.mean())], device=device)

In [82]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_score, accuracy_score
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

In [37]:
model_rnn = RNNClassifier(vocab_size=vocab_size, embed_dim=64, hidden_dim=128, num_layers=1, dropout=0.3).to(device)

In [38]:
class ModelAdapter:

    def forward(self, model, batch):
        raise NotImplementedError

    

In [39]:
class HFModelAdapter(ModelAdapter):

    def forward(self, model, batch):

        output = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])

        logits = output.logits.float().squeeze(1)

        return logits

        

In [40]:
class RNNAdapter(ModelAdapter):

    def forward(self, model, batch):

        logits = model(batch['input_ids'])

        return logits
        



In [ ]:
import copy
class MyTrainer:
    def __init__(
        self, model, name,
        criterion, train_loader, adapter: ModelAdapter,
        valid_loader, test_loader, optimizer,
        scheduler=None, device=None, log_file=None,
        early_stopping=None, thresholds=None,
    ):
        Path(log_file).parent.mkdir(parents=True, exist_ok=True)

        self.logger = logging.getLogger(f"Trainer.{Path(log_file).resolve()}")
        self.logger.setLevel(logging.INFO)
        self.logger.propagate = False

        file_handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
        file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))

        self.logger.addHandler(file_handler)

        self.adapter = adapter
        self.model = model
        self.name = name
        self.criterion = criterion
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.test_loader = test_loader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.early_stopping = early_stopping

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        self.thresholds = (thresholds if thresholds is not None else np.arange(0.01, 1.00, 0.01))

        self.history = []

        self.best_state = None
        self.best_valid_loss = float("inf")
        self.best_ap = float('-inf')

    def train_epoch(self):
        self.model.train()
        train_loss = 0.0

        progress_bar = tqdm(self.train_loader, desc="Training")

        for batch in progress_bar:
            batch = {k: v.to(self.device) for k, v in batch.items()}

            labels = batch["labels"].float()

            self.optimizer.zero_grad(set_to_none=True)

            logits = self.adapter.forward(self.model, batch) 

            assert logits.shape 

            assert torch.isfinite(logits).all(), 'Nan logits'

            loss = self.criterion(logits, labels)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            self.optimizer.step()

            if self.scheduler is not None:
                self.scheduler.step()

            train_loss += loss.item()
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")

        return train_loss / len(self.train_loader)

    @torch.no_grad()
    def validate(self, plot_pr=True, loader=None):
        if loader is None:
            loader = self.valid_loader
        self.model.eval()

        valid_loss = 0.0
        all_probs = []
        all_labels = []

        for batch in tqdm(loader, desc="Validation"):
            batch = {k: v.to(self.device) for k, v in batch.items()}

            labels = batch["labels"].float()

            logits = self.adapter.forward(self.model, batch)

            assert torch.isfinite(logits).all(), 'Nan logits'


            loss = self.criterion(logits, labels)
            valid_loss += loss.item()

            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

        valid_loss/=len(loader)

        all_probs = torch.cat(all_probs).numpy()
        all_labels = torch.cat(all_labels).numpy()

        roc_auc = roc_auc_score(all_labels, all_probs)

        ap = average_precision_score(all_labels, all_probs)
        precision, recall, thresholds = precision_recall_curve(all_labels, all_probs)

        if plot_pr:
            

            plt.figure(figsize=(6, 5))
            plt.plot(recall, precision, label=f"AP = {ap:.3f}")
            plt.axhline(y=all_labels.mean(), color="gray", linestyle="--", label=f"Baseline = {all_labels.mean():.3f}")
            plt.xlabel("Recall")
            plt.ylabel("Precision")
            plt.title("Precision-Recall curve")
            plt.legend()
            plt.grid(True)
            plt.show()

        metrics = []

        for threshold in self.thresholds:
            preds = (all_probs >= threshold).astype(int)

            precision = precision_score(all_labels, preds, zero_division=0)
            recall = recall_score(all_labels, preds, zero_division=0)
            f1 = f1_score(all_labels, preds, zero_division=0)
            acc = accuracy_score(all_labels, preds)


            print(
                f"Name: {self.name} | "
                f"Threshold: {threshold:.3f} | "
                f"Precision: {precision:.3f} | "
                f"Recall: {recall:.3f} | "
                f"F1: {f1:.3f}"
)
        
        return {
            "valid_loss": valid_loss,
            "roc_auc": roc_auc,
            "average_precision": ap,
            "metrics": metrics,
            "probs": np.array(all_probs)
        }


    @torch.no_grad()
    def run_test(self, threshold):


        all_preds = []
        all_labels = []
        all_probs = []
        for batch in tqdm(self.test_loader, desc="Testing"):
            batch = {k: v.to(self.device) for k, v in batch.items()}

            labels = batch["labels"].float()
            
            logits = self.adapter.forward(self.model, batch)

            assert torch.isfinite(logits).all(), 'Nan logits'

            probs = torch.sigmoid(logits)

            preds = (probs >= threshold).long()
            
            all_preds.append(preds.cpu())

            all_labels.append(labels.cpu())

            all_probs.append(probs.cpu())

        all_labels = torch.cat(all_labels).numpy()
        
        all_preds = torch.cat(all_preds).numpy()

        all_probs = torch.cat(all_probs).numpy()

        precision = precision_score(all_labels, all_preds, zero_division=0)
        recall = recall_score(all_labels, all_preds, zero_division=0)
        f1 = f1_score(all_labels, all_preds, zero_division=0)
        acc = accuracy_score(all_labels, all_preds)

        self.logger.critical(f'TEST RESULT: precision: {precision:.3f} | recall: {recall:.4f} | f1: {f1:.3f} | accuracy: {acc:.3f}')


        return {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": acc,
        "probs": np.array(all_probs)
        }

    

    def fit(self, epochs,
        plot_pr=True, print_all_thresholds=True,
        restore_best=True, patience=2
    ):
        for epoch in range(epochs):
            print(f"\nEpoch {epoch + 1}/{epochs}")

            train_loss = self.train_epoch()
            val = self.validate(plot_pr=plot_pr)
            self.logger.warning(f'Name: {self.name} | Validation loss: {val['valid_loss']:.3f} | Validation roc-auc: {val['roc_auc']:.3f}')

            valid_loss = val['valid_loss']
            ap = val['average_precision']
            
            if valid_loss < self.best_valid_loss:
                self.best_valid_loss = valid_loss
                self.best_state = copy.deepcopy(
                    self.model.state_dict()
                )
                patience_counter=0
            else:
                patience_counter+=1

            if ap > self.best_ap:
                self.best_ap = ap

            if patience_counter >= patience:
                print(f'Early stop triggered after {epoch} epoch')
                break

            self.history.append({'validate_metrics': val, 'train_loss': train_loss})
            
            

        if restore_best and self.best_state is not None:
            self.model.load_state_dict(self.best_state)
            print("Restored best model by validation loss.")
            print(f'Best validation loss: {self.best_valid_loss:.3f}')


        return self.history


In [42]:
!uv pip install emotag

Using Python 3.12.12 environment at: /home/ydlawq/torch_rocm_v/.venv
  × No solution found when resolving dependencies:                                  
  ╰─▶ Because emotag was not found in the package registry and you require
      emotag, we can conclude that your requirements are unsatisfiable.


In [83]:
from transformers import AutoModel

In [ ]:
model_id = "deepvk/deberta-v1-distill"

tokenizer = AutoTokenizer.from_pretrained(model_id)
#model = AutoModelForSequenceClassification.from_pretrained(model_id, torch_dtype=torch.float32, num_labels=1, ignore_mismatched_sizes=True, problem_type='multi_label_classification')
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [85]:
encoder_train_ds = MyDataset(X_train['text'], labels=y_train, tokenizer=tokenizer)
encoder_valid_ds = MyDataset(X_valid['text'], labels=y_valid, tokenizer=tokenizer)
encoder_train_loader = DataLoader(encoder_train_ds, collate_fn=data_collator, batch_size=8, shuffle=True) #sampler=sampler)
encoder_valid_loader = DataLoader(encoder_valid_ds, collate_fn=data_collator, batch_size=8)
encoder_test_ds = MyDataset(X_test['text'], labels=y_test, tokenizer=tokenizer)
encoder_test_loader = DataLoader(encoder_test_ds, collate_fn=data_collator, batch_size=8)


In [46]:
import optuna

In [ ]:
def objective(trial):
    lr = trial.suggest_float(
        "lr", 3e-6, 1e-5, log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay", 1e-5, 0.1, log=True
    )

    betas = trial.suggest_float(
        "beta1", 0.85, 0.95
    )

    model = AutoModelForSequenceClassification.from_pretrained(model_id, torch_dtype=torch.float32, num_labels=1, ignore_mismatched_sizes=True, problem_type='multi_label_classification').to(device)
    model.config.pad_token_id = tokenizer.pad_token_id

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, eps=1e-8, weight_decay=weight_decay, betas=(betas, 0.999))
    scheduler = torch.optim.lr_scheduler.LinearLR(optimizer,start_factor=0.3, end_factor=1.0, total_iters=len(encoder_train_loader))

    trainer = MyTrainer(model=model, name='Test', log_file='Test.log',criterion=torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight), scheduler=scheduler,
                    valid_loader=encoder_valid_loader, train_loader=encoder_train_loader,adapter = HFModelAdapter(), test_loader=encoder_test_loader, optimizer=optimizer, device=device)

    trainer.fit(epochs=1)

    

    return trainer.best_ap

In [ ]:
study = optuna.create_study(
    direction="maximize"    
)

study.optimize(
    objective,
    n_trials=15
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_id, torch_dtype=torch.float32, num_labels=1, ignore_mismatched_sizes=True, problem_type='multi_label_classification').to(device)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=study.best_params['lr'], eps=1e-8, weight_decay=study.best_params['weight_decay'], betas=(study.best_params['betas'], 0.999))
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer,start_factor=0.3, end_factor=1.0, total_iters=len(encoder_train_loader))

In [ ]:
trainer = MyTrainer(model=model, name='Test', log_file='Test.log',criterion=torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight), scheduler=scheduler,
                    valid_loader=encoder_valid_loader, train_loader=encoder_train_loader,adapter = HFModelAdapter(), test_loader=encoder_test_loader, optimizer=optimizer, device=device)
